In [ ]:
%matplotlib widget

In [ ]:
import matplotlib as mpl
import matplotlib.typing as mtype
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import time
import datetime
_today = datetime.datetime.today()
while _today.weekday() > 4:  # 0 is Monday, 6 is Sunday
    _today -= datetime.timedelta(days=1)
import pandas as pd
import numpy as np
# from scipy.ndimage import gaussian_filter1d
# import antropy as ant
import logging
import collections
import os, sys, inspect

In [ ]:
%config InlineBackend.figure_format = 'retina'

In [ ]:
parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    # sys.path.append(parent_dir)
    sys.path.insert(0, parent_dir) # prepend
    # print(f"{parent_dir} added to sys.path")


In [ ]:
import ib_insync
from ib_insync import Stock, IB, util
inspect.getfile(ib_insync)


In [ ]:
util.startLoop()

In [ ]:
sym = 'MSFT'

# date = f'{datetime.datetime.now()+datetime.timedelta(days=-1):%Y%m%d}'
date = f'{_today+datetime.timedelta(days=-2):%Y%m%d}'
filename = f'{sym}_1m_{date}.csv'
load_from_csv = True
bars = None
if load_from_csv:
    bars_df = pd.read_csv(rf'.\data\{filename}', parse_dates=['date'])
if 'open_' in bars_df.columns:
    bars_df.rename(columns={'open_': 'open'}, inplace=True)
if 'timestamp' in bars_df.columns:
    bars_df.drop(columns=['timestamp'], inplace=True)


In [ ]:

bars_df['c_m_o'] = bars_df['close'] - bars_df['open']
# volume weighted c_m_o
bars_df['v_c_m_o'] = bars_df['volume'] * bars_df['c_m_o']
# cumulative sum of volume weighted c_m_o
bars_df['v_c_m_o_cumsum'] = bars_df['v_c_m_o'].cumsum()
bars_df['h_m_h'] = bars_df['high'].diff()
bars_df['l_m_l'] = bars_df['low'].diff()
bars_df['c_m_c'] = bars_df['close'].diff()
bars_df['o_m_o'] = bars_df['open'].diff()


In [ ]:
# Filter bars_df for the time range from 9:30am to 4:00pm
m0 = bars_df['date'].dt.time > datetime.time(9, 30)
m99 = bars_df['date'].dt.time < datetime.time(16, 0)
m1 = m0 & m99
bars_df = bars_df[m1].reset_index(drop=True)
bars_df = bars_df[0:300]
x_axis_values = bars_df['date']


In [ ]:

# cumulative sum of positive c_m_o 
bars_df['c_m_o_pos'] = bars_df['c_m_o'].clip(lower=0).cumsum()
bars_df['c_m_o_pos_10'] = bars_df['c_m_o'].clip(lower=0).rolling(window=10).sum()
# cumsum of volume-weighted positive c_m_o
bars_df['v_c_m_o_pos'] = bars_df['v_c_m_o'].clip(lower=0).cumsum()
bars_df['v_c_m_o_pos_10'] = bars_df['v_c_m_o'].clip(lower=0).rolling(window=10).sum()
# cumulative sum of negative c_m_o
bars_df['c_m_o_neg'] = bars_df['c_m_o'].clip(upper=0).cumsum()
bars_df['c_m_o_neg_10'] = bars_df['c_m_o'].clip(upper=0).rolling(window=10).sum()
# cumsum of volume-weighted negative c_m_o
bars_df['v_c_m_o_neg'] = bars_df['v_c_m_o'].clip(upper=0).cumsum()
bars_df['v_c_m_o_neg_10'] = bars_df['v_c_m_o'].clip(upper=0).rolling(window=10).sum()


In [ ]:
bars_df

In [ ]:
# x_axis_values = pd.to_datetime(bars_df.date).dt.tz_localize(None).values
x_axis_values = bars_df.date

In [ ]:
%matplotlib notebook

In [ ]:
fig, axs = plt.subplots(8, 1, sharex=True, figsize=(32, 8), height_ratios=[6, 1, 1, 1, 1, 1, 1, 1])
util.barplot_ohlc(bars_df[['open', 'close', 'high', 'low']], 'Bar plots', upColor='green', downColor='red', fig_ax=(fig, axs[0]))
axs[0].grid(True)

axs[1].plot(bars_df['c_m_o_pos'], label='cumsum of positive c_m_o')
axs[1].plot(bars_df['c_m_o_neg'] * -1, label='cumsum of negative c_m_o')
axs[2].plot(bars_df['c_m_o_pos'] + bars_df['c_m_o_neg'], label='cumsum of positive c_m_o - cumsum of negative c_m_o')
axs[2].axhline(y=0, color='black', linestyle='--', alpha=0.5)

axs[3].plot(bars_df['v_c_m_o_pos'], label='cumsum of positive c_m_o')
axs[3].plot(bars_df['v_c_m_o_neg'] * -1, label='cumsum of negative c_m_o')
axs[4].plot(bars_df['v_c_m_o_pos'] + bars_df['v_c_m_o_neg'], label='cumsum of positive c_m_o - cumsum of negative c_m_o')
axs[4].axhline(y=0, color='black', linestyle='--', alpha=0.5)

axs[4].set_xticks(np.arange(len(x_axis_values)))
axs[4].set_xticklabels(x_axis_values.dt.strftime('%H:%M'))
axs[4].xaxis.set_major_locator(ticker.MultipleLocator(30))
axs[4].grid(True)

# for ax in axs.flat:
#     ax.set_xticks(np.arange(len(x_axis_values)))
#     ax.set_xticklabels(x_axis_values.dt.strftime('%H:%M'))
#     ax.xaxis.set_major_locator(ticker.MultipleLocator(30))
#     ax.grid(True)

axs[5].plot(bars_df['c_m_o_pos_10'], label='cumsum of positive c_m_o')
axs[5].plot(bars_df['c_m_o_neg_10'] * -1, label='cumsum of negative c_m_o')
axs[5].grid(True)

axs[6].plot(bars_df['v_c_m_o_pos_10'], label='cumsum of positive c_m_o')
axs[6].plot(bars_df['v_c_m_o_neg_10'] * -1, label='cumsum of negative c_m_o')
axs[6].grid(True)

axs[7].plot(bars_df['v_c_m_o_pos_10'] + bars_df['v_c_m_o_neg_10'], label='cumsum of positive c_m_o - cumsum of negative c_m_o')
axs[7].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axs[7].grid(True)

plt.legend()
plt.show()
